# 🎬 AMAZON — Cinematic Motion-Graphics Renderer (v3 "Field Journal")

نسخة سينمائية جديدة بالكامل للمحرك. شغّل الخلايا **بالترتيب**.

**قبل البدء:** من القائمة `Runtime ▸ Change runtime type` اختار **T4 GPU** (لازم عشان تقدير العمق ثلاثي الأبعاد يشتغل بسرعة؛ لو مش متاح هيشتغل بطريقة بديلة أبسط).

| الخلية | الوظيفة |
|---|---|
| 1 | تجهيز Drive والمكتبات |
| 2 | تحديد الصور والصوت تلقائياً |
| 3 | تحميل المحرك |
| 4 | **فحص الصور**: قراءة رقم المشهد المطبوع على كل صورة + لوحة مراجعة |
| 5 | **عينة سريعة** من مشاهد مختارة (تظهر هنا للمشاهدة) |
| 6 | **التصدير النهائي 1080p** (يكمّل من حيث توقف لو انقطع Colab) |

In [ ]:
# 1) Drive + المكتبات
from google.colab import drive
drive.mount('/content/drive')
!pip -q install opencv-python-headless arabic-reshaper python-bidi
!apt-get -qq update > /dev/null && apt-get -qq install -y ffmpeg libraqm0 libfribidi0 > /dev/null
import torch, subprocess, shutil
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'غير متاح — سيُستخدم عمق تقريبي')
print('ffmpeg:', shutil.which('ffmpeg'))
print('Arabic shaping (RAQM):', subprocess.run(['python','-c','from PIL import features;print(features.check("raqm"))'],capture_output=True,text=True).stdout.strip())

In [ ]:
# 2) تحديد ملفات المشروع (مجلد رحيم: الصور + الصوت)
import pathlib, os
ROOT = pathlib.Path('/content/drive/MyDrive')
PROJECT = ROOT / 'رحيم'
if not PROJECT.is_dir():
    # المجلد مشترك معك؟ أضف له اختصاراً في My Drive (Add shortcut to Drive) ثم أعد التشغيل
    hits = [p for p in ROOT.rglob('الصور') if p.is_dir()]
    if not hits:
        raise FileNotFoundError('لم أجد مجلد رحيم/الصور. لو المجلد مشترك معك اعمل له Add shortcut to Drive داخل My Drive.')
    PROJECT = hits[0].parent
IMAGES = PROJECT / 'الصور'
audio_dir = PROJECT / 'الصوت'
cands = sorted([p for p in audio_dir.rglob('*') if p.suffix.lower() in ('.wav', '.mp3', '.m4a')],
               key=lambda p: ('generated audio' not in p.name.lower(), p.suffix.lower() != '.wav', p.name))
if not cands:
    raise FileNotFoundError(f'لم أجد ملف صوت داخل {audio_dir}')
AUDIO = cands[0]
WORK = PROJECT / 'Amazon_render_work'          # على Drive: يحفظ المشاهد المنتهية للاستكمال
WORK.mkdir(exist_ok=True)
n_img = sum(1 for q in IMAGES.rglob('*') if q.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp'))
print('الصور :', IMAGES, '| العدد:', n_img)
print('الصوت :', AUDIO)
print('العمل :', WORK)
# ملف مشاهد مخصص (اختياري). المحرك يحتوي تقسيمة الـ90 مشهداً من الـ storyboard بالفعل.
MANIFEST = next((p for p in [PROJECT / 'scenes_manifest_cinematic.json', ROOT / 'scenes_manifest_cinematic.json',
                             ROOT / 'scenes_manifest.json'] if p.exists()), None)
print('المشاهد:', MANIFEST or 'المدمج داخل المحرك (90 مشهداً من الـ storyboard)')

In [ ]:
# 3) المحرك: يُحمَّل من GitHub تلقائياً، أو من مجلد المشروع، أو ارفعه يدوياً
import urllib.request
ENGINE = pathlib.Path('/content/render_amazon_documentary.py')
URL = 'https://raw.githubusercontent.com/rohaim1/catalog/ccr-c87ca09a-33marr/amazon-documentary/render_amazon_documentary.py'
ok = False
local = PROJECT / 'render_amazon_documentary.py'
if local.exists():
    shutil.copy2(local, ENGINE); ok = True; print('المحرك من مجلد المشروع')
else:
    try:
        urllib.request.urlretrieve(URL, ENGINE); ok = ENGINE.stat().st_size > 50000; print('المحرك من GitHub')
    except Exception as e:
        print('GitHub غير متاح:', e)
if not ok:
    from google.colab import files
    print('ارفع ملف render_amazon_documentary.py')
    up = files.upload()
    name = next(n for n in up if n.endswith('.py'))
    shutil.copy2('/content/' + name, ENGINE)
out = subprocess.run(['python', str(ENGINE), '--help'], capture_output=True, text=True)
print('✓ المحرك جاهز' if out.returncode == 0 else out.stderr[-2000:])

def render(*extra):
    """تشغيل المحرك مع عرض التقدم مباشرة."""
    cmd = ['python', '-u', str(ENGINE), '--images', str(IMAGES), '--audio', str(AUDIO), '--work-dir', str(WORK)]
    if MANIFEST: cmd += ['--manifest', str(MANIFEST)]
    cmd += list(extra)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError('فشل التشغيل — راجع الرسالة بالأعلى')

In [ ]:
# 4) فحص الصور: يقرأ رقم المشهد المطبوع على كل صورة (OCR) ويطابقه بالمشهد، ثم يرسم لوحة مراجعة
render('--check', '--output', str(WORK / 'check.mp4'))
from IPython.display import Image, display
display(Image(filename=str(WORK / 'contact_sheet.jpg'), width=1400))
print('أخضر = مطابق بالرقم المطبوع | برتقالي = مشهد بلا صورة فاستُعيرت صورة قريبة بإطار مختلف | أحمر = تخمين بالاسم (راجعه)')

In [ ]:
# 5) عينة سريعة للمراجعة (960x540، حوالي 5 دقائق): الافتتاح + ثعبان الماء + الليل + المطر + الخريطة + النهاية
SAMPLE = WORK / 'Amazon_sample.mp4'
render('--scenes', '1-5,17-18,44,47-48,66-67,89-90', '--width', '960', '--output', str(SAMPLE))
from IPython.display import Video
Video(str(SAMPLE), embed=True, width=960)

In [ ]:
# 6) التصدير النهائي 1920x1080 (حوالي ساعة على Colab) — لا تغلق الصفحة. لو انقطع Colab: أعد تشغيل الخلايا 1→3 ثم هذه الخلية وسيكمل من آخر مشهد.
FINAL = PROJECT / 'Amazon_Documentary_Final.mp4'
render('--output', str(FINAL))
print('✓ الفيديو النهائي:', FINAL)

## إعدادات مفيدة
- `render('--output', str(FINAL), '--titles', 'chapters')` ← عناوين الفصول فقط بدون عنوان لكل مشهد.
- `'--no-sfx'` لإلغاء المؤثرات الصوتية، أو `'--sfx-db', '-4'` لخفضها 4dB.
- `'--music', '/content/drive/MyDrive/music.mp3'` لإضافة موسيقى تنخفض تلقائياً تحت التعليق الصوتي.
- `'--fps', '30'` لـ 30 إطار/ثانية.
- لتعديل أي مشهد (الكاميرا، المؤثرات، الانتقال، النصوص): `render('--check', '--output', str(WORK/'x.mp4'), '--export-manifest', str(PROJECT/'scenes_manifest_cinematic.json'))` ثم عدّل الملف وأعد التشغيل — المحرك يعيد رندر المشاهد المتغيرة فقط.